# 06: sam U-Net (bez RAFT i bez geometrii) na zbiorze z nagrań

Do tej pory U-Net był trenowany tylko na starym zbiorze `rs_dataset_256` (zdjęcia DIV2K, sam ruch kamery):
`2026-10-13_flow_dual` dał 20,45 dB przy wejściu 19,66 dB. Tu ten sam typ sieci uczymy na **`rs_video_x4k`**
(ruch kamery i obiektów), żeby mieć porównanie z RAFT + geometrią z 04 na tym samym zbiorze.

**Model** (`PlainUNet` w `src/raft_unet.py`, 2,7 M parametrów, ta sama architektura U-Net co `flow_dual`):
* wejście: RS A, RS B (obrócony z powrotem do orientacji A), współrzędne pikseli, czas odczytu każdego piksela w A i w B
  (sieć wie, kiedy który wiersz / kolumna była czytana; stary `flow_dual` tego nie miał),
* wyjście: pole przesunięć dla A (obraz wynikowy = `warp(A)`) + drobna poprawka obrazu,
* strata: L1 obrazu względem wzorca GS na całym obrazie.

| sekcja | co |
|---|---|
| 3 | trening, 30 epok (~15 min na L4, szacunek) |
| 4 | wyniki na val razem ze starym U-Netem ocenionym na nowym zbiorze i RAFT z 04, podział na siłę RS |
| 5 | **test na parze obrazów**: wrzucasz obraz RS z kamery A (poziomy) i z kamery B (pionowy), patrzysz, co zwraca U-Net |

Kod: `src/raft_unet.py`, `src/train_unet.py`. **Przed uruchomieniem zrób commit + push.**

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
if not os.path.exists(UNZ):
    print('rozpakowuję', z, '...')
    !unzip -q "{z}" -d "{UNZ}"
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Nazwa runu
Przy wznawianiu innego dnia wpisz nazwę istniejącego folderu z `runs/`.

In [ ]:
from datetime import date
from pathlib import Path
from IPython.display import Image, display
RUN = f'{date.today()}_unet_plain'
OLD_UNET = P.find_run('flow_dual')          # stary U-Net ze zbioru rs_dataset_256 (do porównania)
print('run:', RUN, '| stary U-Net:', OLD_UNET)

## 3. Trening
Po rozłączeniu: SETUP, sekcje 1–2, potem tutaj `RESUME = True`.

In [ ]:
RESUME = False
resume = f'--resume "{P.RUNS / RUN / "last.pt"}"' if RESUME else ''
!python src/train_unet.py --data "{DATA}" --mode plain --epochs 30 --bs 16 --lr 3e-4 \
    --workers {WORKERS} --out "{P.RUNS}" --name {RUN} {resume}

## 4. Wyniki na val
* `psnr`, `ssim`: obszar, w którym piksel źródłowy jest w kadrze A (jak w 04), `psnr_full`: cały obraz, `*_in`: bez korekcji,
* `flow_dual (stary, na nowym zbiorze)`: stary U-Net bez dotrenowania, czy w ogóle przenosi się na ruch obiektów,
* wiersze RAFT pochodzą z `eval.json` zapisanych w 04.

In [ ]:
import json, pandas as pd
rows, per = [], {}
for name in ['raft_zeroshot', 'raft_geom']:
    ej = P.find_run(name, 'eval.json')
    if ej:
        rows.append({'run': ej.parent.name + ' (04)', **json.load(open(ej))['metrics']})
evals = [(RUN, P.RUNS / RUN / 'best.pt', P.RUNS / RUN / 'eval.json')]
if OLD_UNET:
    evals.append(('flow_dual (stary, na nowym zbiorze)', OLD_UNET, P.RUNS / RUN / 'eval_flow_dual_old.json'))
for label, ck, ej in evals:
    if not Path(ck).exists():
        print('brak', ck); continue
    ps = ej.with_name(ej.stem + '_per_sample.csv')
    !python src/train_unet.py --data "{DATA}" --eval "{ck}" --eval-json "{ej}" --per-sample "{ps}" --bs 16 --workers {WORKERS} > /dev/null
    rows.append({'run': label, **json.load(open(ej))['metrics']})
    per[label] = pd.read_csv(ps)
cols = ['psnr', 'ssim', 'psnr_full', 'psnr_in', 'ssim_in', 'valid', 'ms_per_pair', 'params_M']
tab = pd.DataFrame(rows).set_index('run')
display(tab[[c for c in cols if c in tab]].round(3))

Podział na siłę RS (klasy z `src/analyze_dataset.py`: słaby < 2 px ≤ średni < 6 px ≤ silny).

In [ ]:
VAL_ONLY, ANA = '/content/val_only', '/content/analiza_val'
if not os.path.exists(f'{ANA}/samples.csv'):
    os.makedirs(VAL_ONLY, exist_ok=True)
    if not os.path.exists(f'{VAL_ONLY}/val'):
        os.symlink(f'{DATA}/val', f'{VAL_ONLY}/val')
    !python src/analyze_dataset.py --data {VAL_ONLY} --out {ANA} --workers {WORKERS} > /dev/null
cls = pd.read_csv(f'{ANA}/samples.csv').query('split == "val"')[['sample', 'class', 'disp_p95']]
t = cls.copy()
for run, df in per.items():
    df = df.set_index('sample')
    if 'psnr_in' not in t:
        t = t.join(df[['psnr_in']], on='sample')
    if 'psnr_geom' in df and 'RAFT+geom' not in t:
        t = t.join(df['psnr_geom'].rename('RAFT+geom'), on='sample')
    t = t.join(df['psnr'].rename(run), on='sample')
order = ['slaby', 'sredni', 'silny']
summary = t.drop(columns=['sample']).groupby('class').agg(['mean']).droplevel(1, axis=1).reindex(order)
summary.insert(0, 'n', t.groupby('class').size().reindex(order).fillna(0).astype(int))
display(summary.round(2))
print('PSNR [dB] w obszarze widocznym w A; disp_p95 = średnia siła RS w klasie [px]')

In [ ]:
p = sorted((P.RUNS / RUN).glob('preview_ep*.png'))[-1]
print(p, '\nkolumny: RS A | RS B | U-Net | jego błąd | wzorzec')
display(Image(str(p)))

## 5. Test na parze obrazów
1. Wrzuć dwa obrazy na Drive do **`MyDrive/rs_project/real/test_unet/`**:
   * `rs_A*.png` (lub .jpg): obraz z kamery A, czytanej wierszami (poziomo),
   * `rs_B*.png`: obraz z kamery B, czytanej kolumnami (pionowo), **tak jak zapisała go kamera**, czyli obrócony o 90°,
   * opcjonalnie `gs_A*.png`: wzorzec bez RS (wtedy pokaże się też PSNR) i `meta.json` z czasem odczytu.
2. Bez własnych plików komórka bierze parę testową z repozytorium (`examples/test_pair/`, wygenerowaną skryptem
   `src/make_test_pair.py` z klipu ze sceny walidacyjnej, której sieć nie widziała w treningu).
3. Obrazy muszą pokazywać ten sam kwadratowy wycinek sceny; są skalowane do 256×256.
   `T_MS` to czas odczytu tego wycinka (z `meta.json` albo wpisany ręcznie).

In [ ]:
import cv2, torch, numpy as np, matplotlib.pyplot as plt
import raft_unet; importlib.reload(raft_unet)
from raft_unet import load_model, predict_pair, image_tensor
from train import psnr

TEST_DIR = P.ROOT / 'real' / 'test_unet'
if not sorted(TEST_DIR.glob('rs_A*')):
    TEST_DIR = Path(CODE) / 'examples' / 'test_pair'
first = lambda pat: next(iter(sorted(TEST_DIR.glob(pat))), None)
IMG_A, IMG_B, IMG_GS = first('rs_A*'), first('rs_B*'), first('gs_A*')
meta = json.load(open(TEST_DIR / 'meta.json')) if (TEST_DIR / 'meta.json').exists() else {}
T_MS = meta.get('readout_a_s', 0.030) * 1000       # czas odczytu wycinka [ms]
SYNC_MS = meta.get('sync_offset_s', 0.0) * 1000    # środek odczytu B minus środek odczytu A [ms]
ROT_K = meta.get('b_rot_k', 1)                     # 1 albo -1: kierunek obrotu kamery B
print(f'{TEST_DIR} | A: {IMG_A.name}, B: {IMG_B.name}, wzorzec: {IMG_GS.name if IMG_GS else "-"} '
      f'| T {T_MS:.1f} ms, sync {SYNC_MS:+.2f} ms, rot_k {ROT_K}')

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
img_a, img_b = cv2.imread(str(IMG_A)), cv2.imread(str(IMG_B))
models = {'U-Net (nowy zbiór)': (P.RUNS / RUN / 'best.pt', 'unet_nowy')}
if OLD_UNET:
    models['U-Net (stary zbiór)'] = (OLD_UNET, 'unet_stary')
outs = {}
for label, (ck, _) in models.items():
    model, _ = load_model(str(ck), dev)
    rs_a, rs_b, outs[label] = predict_pair(model, img_a, img_b, T_MS, SYNC_MS, 0.0, ROT_K, device=dev)
gs = image_tensor(cv2.imread(str(IMG_GS)), device=dev) if IMG_GS else None

show = lambda t: t[0].clamp(0, 1).permute(1, 2, 0).cpu().numpy()
panels = [('RS A (poziomy)', rs_a), ('RS B (obrócony do A)', rs_b)] + list(outs.items())
if gs is not None:
    panels.append(('wzorzec GS', gs))
fig, ax = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4.8))
for a_, (title, im) in zip(ax, panels):
    if gs is not None and title not in ('wzorzec GS', 'RS B (obrócony do A)'):
        title += f'\nPSNR {psnr(im, gs, torch.ones_like(gs[:, :1])).item():.2f} dB'
    a_.imshow(show(im)); a_.set_title(title); a_.axis('off')
plt.tight_layout()
OUT = P.RESULTS / 'test_unet'; OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / f'{RUN}.png', dpi=100, bbox_inches='tight')
plt.show()
for label, im in outs.items():
    cv2.imwrite(str(OUT / f'{RUN}_{models[label][1]}.png'),
                (show(im)[..., ::-1] * 255).round().astype(np.uint8))
print('zapisano w', OUT)

## 6. Po treningu
Dopisz wiersz `*_unet_plain` (zbiór `rs_video_x4k`, notatnik 06) do `experiments/README.md` oraz wiersz
„flow_dual oceniony na rs_video_x4k” z tabeli w sekcji 4. Commit notatnika razem z wynikami.